# Banking & Finance Multi-Agent Assistant

This notebook demonstrates a **Multi-Agent Banking & Finance Assistant** built with **LangGraph** and the **Google GenAI SDK**.

### System Architecture
- **State**: Tracks user messages, user session context, mock bank database outputs, and calculated financial summaries.
- **Intent Classifier Node**: Uses `gemini-2.5-flash` to evaluate user input and route the request to the correct specialist agent.
- **Specialized Agents/Nodes**:
  1. **Balance Agent**: Retrieves current checking, savings, and credit card balances.
  2. **Transaction Agent**: Retrieves recent transactions from the customer ledger.
  3. **Loan Calculator Agent**: Performs financial mathematics for EMIs and amortization schedules based on user-provided inputs.
  4. **Investment Advisor Agent**: Offers asset allocation guidelines matching risk profiles and user assets.
  5. **General FAQ Agent**: Handles bank information queries (e.g. routing numbers, business hours).
- **Response Synthesizer**: Reviews findings and formats a secure, professional financial response.

In [ ]:
# Install dependencies (uses --no-warn-script-location to suppress PATH warnings)
# !pip install langgraph google-genai python-dotenv -q

In [11]:
import os
import json
from typing import TypedDict, List, Dict, Any, Optional
from dotenv import load_dotenv
from google import genai

# Load environmental variables from .env file
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    print("❌ ERROR: GOOGLE_API_KEY is not set. Please update the .env file.")
else:
    print("✅ GOOGLE_API_KEY detected successfully.")
    client = genai.Client(api_key=api_key)

✅ GOOGLE_API_KEY detected successfully.


## 1. Mock Banking Database
We define a simulated customer database with checking, savings, credit accounts, and recent transaction logs. The user in our simulation is `user_101` (Alex Mercer).

In [13]:
MOCK_DB = {
    "user_101": {
        "name": "Alex Mercer",
        "accounts": {
            "checking": 5430.75,
            "savings": 48200.50,
            "credit_card": -1250.00
        },
        "transactions": [
            {"date": "2026-07-01", "description": "Grocery Store", "amount": -120.50, "category": "Groceries"},
            {"date": "2026-07-02", "description": "Employer Paycheck", "amount": 2850.00, "category": "Income"},
            {"date": "2026-07-03", "description": "Electric Utility", "amount": -85.20, "category": "Utilities"},
            {"date": "2026-07-04", "description": "Coffee Shop", "amount": -6.75, "category": "Dining"}
        ]
    }
}

## 2. Define the Graph State
Our graph state (`BankingState`) keeps track of user commands, routing targets, databases results, math calculations, and final replies.

In [14]:
class BankingState(TypedDict):
    user_id: str
    user_query: str
    current_intent: Optional[str]
    next_action: Optional[str]
    account_info: Optional[Dict[str, float]]
    transactions: Optional[List[Dict[str, Any]]]
    loan_calculations: Optional[Dict[str, Any]]
    agent_response: Optional[str]

## 3. Implement Agent Nodes
Nodes are modular Python functions performing specific backend actions or interacting with Gemini.

In [15]:
def classifier_node(state: BankingState):
    print("🔍 [Classifier Node] Determining user intent...")
    query = state["user_query"]
    
    prompt = (
        "You are an intent classifier for a banking assistant. "
        "Classify the following user query into exactly one of these categories:\n"
        "- balance_check\n"
        "- transaction_history\n"
        "- loan_calculator\n"
        "- investment_advice\n"
        "- general_qa\n\n"
        f"Query: \"{query}\"\n\n"
        "Return ONLY the classification category string without any punctuation or other words."
    )
    
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    
    intent = response.text.strip().lower()
    valid_intents = ["balance_check", "transaction_history", "loan_calculator", "investment_advice", "general_qa"]
    if intent not in valid_intents:
        intent = "general_qa"
        
    print(f"   Classified Intent: {intent}")
    return {"current_intent": intent, "next_action": intent}

def balance_node(state: BankingState):
    print("💳 [Balance Agent] Fetching account balances...")
    uid = state["user_id"]
    user_data = MOCK_DB.get(uid, {})
    accounts = user_data.get("accounts", {})
    return {"account_info": accounts, "next_action": "responder"}

def transaction_node(state: BankingState):
    print("📜 [Transaction Agent] Fetching transaction ledger...")
    uid = state["user_id"]
    user_data = MOCK_DB.get(uid, {})
    transactions = user_data.get("transactions", [])
    return {"transactions": transactions, "next_action": "responder"}

def loan_node(state: BankingState):
    print("🧮 [Loan Calculator Agent] Parsing loan parameters & computing...")
    query = state["user_query"]
    
    # Ask Gemini to parse the variables from the query into a structured JSON string
    prompt = (
        f"Parse the loan details from this query: \"{query}\"\n\n"
        "Return a valid JSON object (and nothing else) with the keys:\n"
        "- principal (float or null)\n"
        "- annual_rate (float interest rate in percentage, or null)\n"
        "- term_years (float/integer length in years, or null)\n"
        "Example format: {\"principal\": 20000, \"annual_rate\": 6.0, \"term_years\": 5}"
    )
    
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    
    # Simple JSON extraction
    text = response.text.strip()
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0].strip()
    elif "```" in text:
        text = text.split("```")[1].split("```")[0].strip()
        
    calc_results = {"error": "Could not parse loan details. Please specify principal, interest rate, and term length."}
    try:
        params = json.loads(text)
        p = params.get("principal")
        rate = params.get("annual_rate")
        years = params.get("term_years")
        
        if p and rate and years:
            # Perform EMI calculations
            r = rate / (12 * 100) # monthly interest rate
            n = int(years * 12)   # number of monthly payments
            emi = p * r * ((1 + r)**n) / (((1 + r)**n) - 1)
            total_payment = emi * n
            total_interest = total_payment - p
            
            calc_results = {
                "principal": p,
                "annual_rate": rate,
                "term_years": years,
                "monthly_emi": round(emi, 2),
                "total_payment": round(total_payment, 2),
                "total_interest": round(total_interest, 2)
            }
            print(f"   Calculated EMI: {calc_results['monthly_emi']}")
    except Exception as e:
        print(f"   Error parsing loan details: {e}")
        
    return {"loan_calculations": calc_results, "next_action": "responder"}

def investment_node(state: BankingState):
    print("📈 [Investment Advisor Agent] Generating portfolio allocation...")
    uid = state["user_id"]
    user_data = MOCK_DB.get(uid, {})
    checking = user_data.get("accounts", {}).get("checking", 0.0)
    savings = user_data.get("accounts", {}).get("savings", 0.0)
    total_assets = checking + savings
    query = state["user_query"]
    
    prompt = (
        f"The customer has ${total_assets:,.2f} in liquid assets (Savings: ${savings:,.2f}, Checking: ${checking:,.2f}).\n"
        f"They are asking: \"{query}\"\n\n"
        "Provide a tailored, high-level investment advice overview based on their assets. "
        "Suggest a simple asset allocation model (e.g. Stocks/Bonds/Cash percentage split) and list 3 diversification steps. "
        "Keep it informative, noting that this is not official financial advice."
    )
    
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    
    return {"agent_response": response.text.strip(), "next_action": "responder"}

def general_qa_node(state: BankingState):
    print("💬 [General FAQ Agent] Answering bank query...")
    query = state["user_query"]
    
    prompt = (
        "You are a friendly customer service bank teller assistant. "
        f"Answer this query: \"{query}\"\n\n"
        "Ensure your response is secure, polite, professional, and clear."
    )
    
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    
    return {"agent_response": response.text.strip(), "next_action": "responder"}

def responder_node(state: BankingState):
    print("✍️ [Response Synthesizer Node] Formulating final user response...")
    intent = state["current_intent"]
    
    # If agent_response was already generated (e.g., by advisor or FAQ node), return it directly
    if state.get("agent_response"):
        return {"next_action": "end"}
        
    # Otherwise, synthesize state into a clean template response using Gemini
    context_data = {}
    if intent == "balance_check":
        context_data = {"accounts": state.get("account_info")}
    elif intent == "transaction_history":
        context_data = {"transactions": state.get("transactions")}
    elif intent == "loan_calculator":
        context_data = {"loan_results": state.get("loan_calculations")}
        
    prompt = (
        "You are a professional banking assistant. Write a polite, clear response to the customer's request. "
        f"User Query: \"{state['user_query']}\"\n"
        f"Context retrieved from bank databases: {json.dumps(context_data)}\n\n"
        "Structure the details neatly. Use dollar signs and clean formatting. "
        "If transaction records are present, show them in a neat list or table. "
        "If a math error/loan warning is present, explain it politely."
    )
    
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    
    return {"agent_response": response.text.strip(), "next_action": "end"}

## 4. Build and Compile the Graph
We assemble our routing layout, adding edges between classifier, agents, and final response compiler nodes.

In [16]:
from langgraph.graph import StateGraph, START, END

# Create the graph
workflow = StateGraph(BankingState)

# Register nodes
workflow.add_node("classifier", classifier_node)
workflow.add_node("balance_agent", balance_node)
workflow.add_node("transaction_agent", transaction_node)
workflow.add_node("loan_agent", loan_node)
workflow.add_node("investment_agent", investment_node)
workflow.add_node("general_agent", general_qa_node)
workflow.add_node("responder", responder_node)

# Define routing rules
workflow.add_edge(START, "classifier")

# Define conditional routing from classifier node
def route_from_classifier(state: BankingState):
    return state["next_action"]

workflow.add_conditional_edges(
    "classifier",
    route_from_classifier,
    {
        "balance_check": "balance_agent",
        "transaction_history": "transaction_agent",
        "loan_calculator": "loan_agent",
        "investment_advice": "investment_agent",
        "general_qa": "general_agent"
    }
)

# Add edges from agents to responder node
workflow.add_edge("balance_agent", "responder")
workflow.add_edge("transaction_agent", "responder")
workflow.add_edge("loan_agent", "responder")
workflow.add_edge("investment_agent", "responder")
workflow.add_edge("general_agent", "responder")

# Define end condition
def route_from_responder(state: BankingState):
    if state["next_action"] == "end":
        return END
    return END

workflow.add_conditional_edges(
    "responder",
    route_from_responder,
    {
        END: END
    }
)

# Compile the graph
app = workflow.compile()
print("🎉 Banking Finance Multi-Agent Graph compiled successfully!")

🎉 Banking Finance Multi-Agent Graph compiled successfully!


## 5. Execution Demonstration
We now test our compiled banking assistant with various user query inputs. Notice how LangGraph routes each request to its specialized agent, computes outputs, and formats responses dynamically.

In [17]:
def test_assistant(query_str: str):
    print(f"\nInput: \"{query_str}\"")
    initial_state = {
        "user_id": "user_101",
        "user_query": query_str
    }
    
    result = app.invoke(initial_state)
    print("\n=== FINAL RESPONSE ===")
    print(result.get("agent_response"))
    print("=" * 30)

# Scenario 1: Checking Account Balance
test_assistant("Hey, can you tell me what my checking and savings account balances look like right now?")


Input: "Hey, can you tell me what my checking and savings account balances look like right now?"
🔍 [Classifier Node] Determining user intent...
   Classified Intent: balance_check
💳 [Balance Agent] Fetching account balances...
✍️ [Response Synthesizer Node] Formulating final user response...

=== FINAL RESPONSE ===
Hello! I can certainly help you with your account balances.

Here are your current balances as of right now:

*   **Checking Account:** $5,430.75
*   **Savings Account:** $48,200.50

If you need any further assistance or have more questions, please don't hesitate to ask!


In [18]:
# Scenario 2: Transaction History
test_assistant("Show me my recent transactions list.")


Input: "Show me my recent transactions list."
🔍 [Classifier Node] Determining user intent...
   Classified Intent: transaction_history
📜 [Transaction Agent] Fetching transaction ledger...
✍️ [Response Synthesizer Node] Formulating final user response...

=== FINAL RESPONSE ===
Certainly! Here are your recent transactions:

*   **2026-07-01:** Grocery Store: **-$120.50** (Groceries)
*   **2026-07-02:** Employer Paycheck: **+$2,850.00** (Income)
*   **2026-07-03:** Electric Utility: **-$85.20** (Utilities)
*   **2026-07-04:** Coffee Shop: **-$6.75** (Dining)

If you need more details on a specific transaction or would like to see an older transaction history, please let me know.


In [19]:
# Scenario 3: Loan EMI Calculation
test_assistant("If I borrow $20000 for a car loan at 6% interest for 5 years, how much will I pay per month?")


Input: "If I borrow $20000 for a car loan at 6% interest for 5 years, how much will I pay per month?"
🔍 [Classifier Node] Determining user intent...
   Classified Intent: loan_calculator
🧮 [Loan Calculator Agent] Parsing loan parameters & computing...
   Calculated EMI: 386.66
✍️ [Response Synthesizer Node] Formulating final user response...

=== FINAL RESPONSE ===
Certainly! I can help you with that calculation for your car loan.

Based on the details you provided, here's what your monthly payment would be:

**Your estimated monthly payment will be $386.66.**

Here's a breakdown of the loan details:

*   **Loan Amount (Principal):** $20,000.00
*   **Annual Interest Rate:** 6.00%
*   **Loan Term:** 5 years (or 60 months)
*   **Estimated Monthly Payment (EMI):** $386.66
*   **Total Interest Paid Over Loan Term:** $3,199.36
*   **Total Amount Repaid Over Loan Term:** $23,199.36

Please note that this calculation is an estimate and actual payments may vary slightly due to factors like ex

In [20]:
# Scenario 4: Investment Advice based on assets
test_assistant("I have a moderate risk preference. How should I distribute my money?")


Input: "I have a moderate risk preference. How should I distribute my money?"
🔍 [Classifier Node] Determining user intent...
   Classified Intent: investment_advice
📈 [Investment Advisor Agent] Generating portfolio allocation...
✍️ [Response Synthesizer Node] Formulating final user response...

=== FINAL RESPONSE ===
Okay, let's look at your situation with a moderate risk preference. You have a solid $53,631.25 in liquid assets, which is a great foundation to start building your wealth.

Given your moderate risk preference, the goal is to balance growth potential with a reasonable level of stability. This typically means a mix skewed towards growth assets (like stocks) but also including assets that provide more stability (like bonds) and maintaining some readily available cash.

***

### Tailored, High-Level Investment Advice Overview

With your current assets entirely in cash, you have a significant opportunity to put your money to work for you. A moderate risk tolerance suggests yo

In [21]:
# Scenario 5: General Q&A
test_assistant("What is the transit routing number for First National Bank?")


Input: "What is the transit routing number for First National Bank?"
🔍 [Classifier Node] Determining user intent...
   Classified Intent: general_qa
💬 [General FAQ Agent] Answering bank query...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 42.060179078s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},